[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/SFM/blob/main/Quantlets/Ch_10/SFM_ch10_portfolio_copula/SFM_ch10_portfolio_copula.ipynb)

# SFM_ch10_portfolio_copula

A 50/50 portfolio of the BET and the S&P 500 (prices joined on common days, simple returns): variance-covariance VaR 1% and ES 2.5%, diversification benefit and Euler contributions; 250-day rolling correlation; Kendall tau, empirical lower tail dependence; Gaussian and t copulas (rho from Kendall tau, nu by maximum likelihood), their tail dependence; portfolio VaR and ES by copula simulation with empirical margins against historical simulation.

*Statistics of Financial Markets (SFM), Chapter 10: VaR, ES and backtesting. Author: Daniel Traian Pele.*

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/SFM/Chapter_10'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
# the arch package (maximum-likelihood estimation of GARCH models) is not part of Google Colab
!pip install -q arch

In [ ]:
import os
import re
import json
import urllib.request
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb
from scipy import optimize
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

## Chart style

In [ ]:
# Palette (RGB values of latex/preamble.tex)
MainBlue = '#1A3A6E'
IDAred = '#CD0000'
Forest = '#2E7D32'
Amber = '#B5853F'
Orange = '#E67E22'
Purple = '#8E44AD'
Teal = '#17A2B8'
Crimson = '#DC3545'
DarkText = '#1F2A44'          # text, axes and ticks (dark navy, not grey)

PALETTE = [MainBlue, IDAred, Forest, Amber, Purple, Orange, Teal, Crimson]
# fixed colours for the series used in several chapters
COL = {'sp500': MainBlue, 'bet': IDAred, 'bettr': Orange, 'dax': Teal, 'btc': Amber, 'eth': Crimson,
       'eurron': Forest, 'gold': Purple, 'vix': Crimson, 'stoxx50': Forest, 'ndx': Teal}


def apply():
    """Set the course style for matplotlib."""
    rc = plt.rcParams
    rc['figure.facecolor'] = 'none'
    rc['axes.facecolor'] = 'none'
    rc['savefig.facecolor'] = 'none'
    rc['savefig.transparent'] = True
    rc['axes.grid'] = False
    rc['font.family'] = 'sans-serif'
    rc['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
    # font sizes for slides (charts are 9-11 inches wide and shown at 8-14 cm)
    rc['font.size'] = 12
    rc['axes.labelsize'] = 13
    rc['axes.titlesize'] = 13
    rc['xtick.labelsize'] = 11.5
    rc['ytick.labelsize'] = 11.5
    rc['legend.fontsize'] = 11
    rc['axes.spines.top'] = False
    rc['axes.spines.right'] = False
    rc['axes.linewidth'] = 0.6
    rc['lines.linewidth'] = 1.4
    rc['legend.facecolor'] = 'none'
    rc['legend.framealpha'] = 0
    rc['legend.frameon'] = False
    for k in ('text.color', 'axes.labelcolor', 'axes.edgecolor', 'xtick.color', 'ytick.color', 'axes.titlecolor'):
        rc[k] = DarkText
    rc['axes.prop_cycle'] = mpl.cycler(color=PALETTE)


def legend_outside_bottom(ax, ncol=2, y=-0.22, **kw):
    """Place the legend outside the plot, bottom centre (for a figure with several axes, pass the last one
    or use fig.legend with the same arguments)."""
    return ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False, **kw)


def fig_legend_bottom(fig, handles=None, labels=None, ncol=3, y=-0.02):
    """One legend for a whole figure (several panels), below the panels."""
    if handles is None:
        handles, labels = [], []
        for ax in fig.axes:
            for h, l in zip(*ax.get_legend_handles_labels()):
                if l not in labels and not l.startswith('_'):
                    handles.append(h)
                    labels.append(l)
    return fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def _is_grey(c, tol=0.06):
    try:
        r, g, b = to_rgb(c)
    except ValueError:
        return False
    return max(r, g, b) - min(r, g, b) < tol and 0.25 < (r + g + b) / 3 < 0.95


def check_no_grey(fig):
    """House rule: no grey series and no grey text. Raises ValueError listing the offending elements."""
    bad = []
    for ax in fig.axes:
        for ln in ax.get_lines():
            if not ln.get_label().startswith('_') and _is_grey(ln.get_color()):
                bad.append(f'line {ln.get_label()!r}')
        for coll in ax.collections:
            fc = coll.get_facecolor()
            if len(fc) and coll.get_label() and not coll.get_label().startswith('_') and _is_grey(fc[0][:3]):
                bad.append(f'series {coll.get_label()!r}')
        for t in ax.texts + [ax.title, ax.xaxis.label, ax.yaxis.label]:
            if t.get_text() and _is_grey(t.get_color()):
                bad.append(f'text {t.get_text()[:30]!r}')
    if bad:
        raise ValueError('grey elements: ' + ', '.join(bad))


def save_fig(name, out_dir=None, show=True):
    """Save the figure as transparent PDF and PNG (OUT_DIR if defined, else the current folder), then show it."""
    d = out_dir or globals().get('OUT_DIR', '.')
    plt.savefig(os.path.join(d, f'{name}.pdf'), bbox_inches='tight', transparent=True)
    plt.savefig(os.path.join(d, f'{name}.png'), bbox_inches='tight', transparent=True, dpi=180)
    if show:
        plt.show()
    plt.close()


apply()
# the chapter scripts call the style as st.<name> (import sfm_style as st): the same names here
import types
st = types.SimpleNamespace(COL=COL, PALETTE=PALETTE, MainBlue=MainBlue, IDAred=IDAred, Forest=Forest, Amber=Amber,
                           Orange=Orange, Purple=Purple, Teal=Teal, Crimson=Crimson, DarkText=DarkText, apply=apply, legend_outside_bottom=legend_outside_bottom,
                           fig_legend_bottom=fig_legend_bottom, save_fig=save_fig, check_no_grey=check_no_grey)

## Data

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/SFM/main/data/market/'
# local copy of the course data, if the notebook runs inside the repository
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
END = '2026-09-18'          # last day of the saved data

# name -> (symbol, label, group, field, default start)
SERIES = {
    # equity indices
    'sp500':    ('GSPC.INDX',     'S&P 500',               'Equity', 'close', '2000-01-01'),
    'ndx':      ('NDX.INDX',      'Nasdaq 100',            'Equity', 'close', '2000-01-01'),
    'dax':      ('GDAXI.INDX',    'DAX',                   'Equity', 'close', '2000-01-01'),
    'stoxx50':  ('STOXX50E.INDX', 'Euro Stoxx 50',         'Equity', 'close', '2000-01-01'),
    'nikkei':   ('N225.INDX',     'Nikkei 225',            'Equity', 'close', '2000-01-01'),
    'bet':      ('BET',           'BET',                   'Equity', 'close', '2000-01-01'),
    'bettr':    ('BETTR.INDX',    'BET-TR',                'Equity', 'close', '2014-09-23'),
    'betxt':    ('BETXT.INDX',    'BET-XT',                'Equity', 'close', '2011-08-12'),
    'betfi':    ('BETFI.INDX',    'BET-FI',                'Equity', 'close', '2012-01-25'),
    'wig20':    ('WIG20.INDX',    'WIG20',                 'Equity', 'close', '2000-01-01'),
    'bux':      ('BUX.INDX',      'BUX',                   'Equity', 'close', '2000-01-01'),
    'px':       ('PX.INDX',       'PX',                    'Equity', 'close', '2000-01-01'),
    'vix':      ('VIX.INDX',      'VIX',                   'Volatility', 'close', '2000-01-01'),
    # ETFs (adjusted close)
    'spy':      ('SPY.US',        'SPY',                   'ETF', 'adjusted_close', '2000-01-01'),
    'tlt':      ('TLT.US',        'TLT (US Treasuries 20y+)', 'ETF', 'adjusted_close', '2002-07-30'),
    'gld':      ('GLD.US',        'GLD (gold ETF)',        'ETF', 'adjusted_close', '2004-11-18'),
    'tvbetetf': ('TVBETETF.RO',   'TVBETETF',              'ETF', 'adjusted_close', '2015-06-12'),
    # Bucharest Stock Exchange (adjusted close)
    'tlv':      ('TLV.RO',        'Banca Transilvania',    'Stock', 'adjusted_close', '2000-01-01'),
    'snp':      ('SNP.RO',        'OMV Petrom',            'Stock', 'adjusted_close', '2001-09-04'),
    'brd':      ('BRD.RO',        'BRD',                   'Stock', 'adjusted_close', '2001-01-16'),
    'tgn':      ('TGN.RO',        'Transgaz',              'Stock', 'adjusted_close', '2008-01-25'),
    'sng':      ('SNG.RO',        'Romgaz',                'Stock', 'adjusted_close', '2013-11-12'),
    'snn':      ('SNN.RO',        'Nuclearelectrica',      'Stock', 'adjusted_close', '2013-11-04'),
    'h2o':      ('H2O.RO',        'Hidroelectrica',        'Stock', 'adjusted_close', '2023-07-12'),
    # US stocks (adjusted close)
    'aapl':     ('AAPL.US',       'Apple',                 'Stock', 'adjusted_close', '2000-01-01'),
    'msft':     ('MSFT.US',       'Microsoft',             'Stock', 'adjusted_close', '2000-01-01'),
    'nvda':     ('NVDA.US',       'NVIDIA',                'Stock', 'adjusted_close', '2000-01-01'),
    'jpm':      ('JPM.US',        'JPMorgan Chase',        'Stock', 'adjusted_close', '2000-01-01'),
    # crypto (close, 7 days a week)
    'btc':      ('BTC-USD.CC',    'Bitcoin',               'Crypto', 'close', '2014-09-17'),
    'eth':      ('ETH-USD.CC',    'Ethereum',              'Crypto', 'close', '2015-08-07'),
    'sol':      ('SOL-USD.CC',    'Solana',                'Crypto', 'close', '2020-04-11'),
    'usdt':     ('USDT-USD.CC',   'Tether (USDT)',         'Crypto', 'close', '2015-02-26'),
    # FX and commodities
    'eurron':   ('REF:EUR',       'EUR/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'usdron':   ('REF:USD',       'USD/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'eurron_eodhd': ('EURRON.FOREX', 'EUR/RON (EODHD)',    'FX', 'close', '2005-07-01'),
    'eurusd':   ('EURUSD.FOREX',  'EUR/USD',               'FX', 'close', '2002-05-06'),
    'gold':     ('XAUUSD.FOREX',  'Gold (XAU/USD)',        'Commodity', 'close', '2000-01-01'),
    # government bond yields (close, % p.a.)
    'us10y':    ('US10Y.GBOND',   'US 10Y yield',          'Yield', 'close', '2000-01-01'),
    'de10y':    ('DE10Y.GBOND',   'Germany 10Y yield',     'Yield', 'close', '2007-03-28'),
    'ro10y':    ('RO10Y.GBOND',   'Romania 10Y yield',     'Yield', 'close', '2007-08-17'),
}
LABELS = {k: v[1] for k, v in SERIES.items()}
_CACHE = {}


def read_market(symbol):
    """Daily table of one symbol: local copy of the course data, otherwise the SFM repository on GitHub."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname) if MARKET_DIR else ''
    src = path if path and os.path.exists(path) else REPO_RAW + fname
    return pd.read_csv(src, index_col='date', parse_dates=True).sort_index()


def read_reference_rate(currency='EUR', start='2005-07-01', end=END):
    """Official BNR reference rate (RON per unit of currency), from the public yearly XML archives."""
    key = (currency, start, end)
    if key in _CACHE:
        return _CACHE[key]
    rows = []
    for y in range(max(int(start[:4]), 2005), int(end[:4]) + 1):
        url = f'https://curs.bnr.ro/files/xml/years/nbrfxrates{y}.xml'
        xml = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla'}),
                                     timeout=60).read().decode()
        for d, body in re.findall(r'<Cube date="([\d-]+)">(.*?)</Cube>', xml, re.S):
            m = re.search(rf'<Rate currency="{currency}"(?: multiplier="\d+")?>([\d.]+)</Rate>', body)
            if m:
                rows.append((d, float(m.group(1))))
    s = pd.DataFrame(rows, columns=['date', 'close']).drop_duplicates('date').set_index('date')['close']
    s.index = pd.to_datetime(s.index)
    _CACHE[key] = s.sort_index().loc[start:end]
    return _CACHE[key]


def load_close(name, start=None, end=END, field=None):
    """Daily closing price of a named series, cleaned with the course conventions."""
    symbol, label, group, field0, start0 = SERIES[name]
    start, field = start or start0, field or field0
    if symbol.startswith('REF:'):
        return read_reference_rate(symbol.split(':')[1], start=start, end=end).rename(name)
    t = read_market(symbol)
    s = (t[field] if field in t.columns else t['close']).loc[start:end]
    s = pd.to_numeric(s, errors='coerce').dropna()
    if group != 'Yield':
        s = s[s > 0]
    if group != 'Crypto':
        s = s[s.index.dayofweek < 5]          # no weekend quotes
        s = s[s.diff() != 0]                  # no holidays filled with the previous price
    s.index.name = 'date'
    return s.rename(name)


def load_ohlc(name, start=None, end=END):
    """Open, high, low, close (and volume) of a series with OHLC data (range-based volatility estimators)."""
    symbol, _, group, _, start0 = SERIES[name]
    t = read_market(symbol).loc[start or start0:end]
    cols = [c for c in ('open', 'high', 'low', 'close', 'volume') if c in t.columns]
    t = t[cols].dropna()
    if group != 'Crypto':
        t = t[t.index.dayofweek < 5]
    return t[(t[['open', 'high', 'low', 'close']] > 0).all(axis=1)] if 'open' in cols else t


def log_returns(name, start=None, end=END, pct=True):
    """Daily log returns r_t = ln P_t - ln P_{t-1} (in % by default), on the series' own calendar."""
    r = np.log(load_close(name, start, end)).diff().dropna()
    return (100 * r if pct else r).rename(name)


def simple_returns(name, start=None, end=END, pct=True):
    """Daily simple returns R_t = P_t / P_{t-1} - 1 (in % by default)."""
    r = load_close(name, start, end).pct_change().dropna()
    return (100 * r if pct else r).rename(name)


def load_panel(names, start=None, end=END, kind='close'):
    """Several series aligned on date: kind = 'close' or 'returns' (NaN where a market does not trade)."""
    f = load_close if kind == 'close' else log_returns
    return pd.concat([f(n, start, end) for n in names], axis=1)


def periods_per_year(r):
    """Average number of observations per calendar year (the actual frequency of the series)."""
    years = (r.index[-1] - r.index[0]).days / 365.25
    return len(r) / years

## Analysis

In [ ]:
from arch import arch_model

NAME = {'sp500': 'S&P 500', 'dax': 'DAX', 'bet': 'BET', 'btc': 'Bitcoin', 'tlv': 'Banca Transilvania', 'snp': 'OMV Petrom', 'brd': 'BRD'}

START = {'sp500': '2000-01-01', 'dax': '2000-01-01', 'bet': '2000-01-01', 'btc': None, 'tlv': '2010-01-01', 'snp': '2010-01-01', 'brd': '2010-01-01'}

ASSETS = ['sp500', 'dax', 'bet', 'btc', 'tlv', 'snp']

BT_ASSETS = ['sp500', 'dax', 'bet', 'btc']

BAD_DAYS = {'tlv': ['2016-05-30', '2016-05-31']}

COLORS = {'sp500': '#1A3A6E', 'dax': '#17A2B8', 'bet': '#CD0000', 'btc': '#B5853F', 'tlv': '#E67E22', 'snp': '#2E7D32'}

EPISODES = {'2008': ('2008-09-01', '2009-03-31'), '2020': ('2020-02-15', '2020-05-31')}

ALPHA_VAR = 0.01

ALPHA_ES = 0.025

WINDOW = 500

REFIT = 250

OOS_START = {'sp500': '2005-01-01', 'dax': '2005-01-01', 'bet': '2005-01-01', 'btc': '2017-01-01'}

POT_Q = 0.9

METHODS = ['HS', 'Normal', 'GARCH-t', 'FHS']

MCOL = {'HS': '#B5853F', 'Normal': '#2E7D32', 'GARCH-t': '#CD0000', 'FHS': '#1A3A6E'}

PORT = ('bet', 'sp500')

WEIGHTS = (0.5, 0.5)

N_SIM = 200000

N_Z2 = 1000

SEED = 2026

PLUS = {0: 0.0, 1: 0.0, 2: 0.0, 3: 0.0, 4: 0.0, 5: 0.4, 6: 0.5, 7: 0.65, 8: 0.75, 9: 0.85}

def returns(k, start=None, end=None):
    """Daily log returns in % on the series' own calendar; known data errors removed."""
    r = log_returns(k, start or START.get(k)) if end is None else log_returns(k, start or START.get(k), end)
    if k in BAD_DAYS:
        r = r.drop(pd.to_datetime(BAD_DAYS[k]), errors='ignore')
    return r


def hs_var(x, a=ALPHA_VAR):
    """Historical simulation: VaR_a = -r_(k), the k-th smallest return, k = ceil(n a)."""
    xs = np.sort(np.asarray(x, float))
    k = int(np.ceil(len(xs) * a - 1e-9))
    return float(-xs[k - 1])


def hs_es(x, a=ALPHA_ES):
    """Historical simulation: ES_a = minus the average of the k smallest returns, k = ceil(n a)."""
    xs = np.sort(np.asarray(x, float))
    k = int(np.ceil(len(xs) * a - 1e-9))
    return float(-xs[:k].mean())


def normal_var(mu, sd, a=ALPHA_VAR):
    """VaR_a = -(mu + sd z_a), z_a the a-quantile of N(0, 1) (negative)."""
    return float(-(mu + sd * stats.norm.ppf(a)))


def normal_es(mu, sd, a=ALPHA_ES):
    """ES_a = -mu + sd phi(z_a) / a."""
    return float(-mu + sd * stats.norm.pdf(stats.norm.ppf(a)) / a)


def portfolio_data(keys=PORT):
    """Prices joined on common days first, then simple returns in % (the portfolio return is linear in them)."""
    P = pd.concat([load_close(k, START.get(k)) for k in keys], axis=1).dropna()
    R = 100 * P.pct_change().dropna()
    R.columns = list(keys)
    return R


def pseudo_obs(R):
    """Ranks divided by n + 1: the empirical copula sample (uniform margins)."""
    return R.rank().values / (len(R) + 1)


def t_copula_loglik(U, rho, nu):
    """Log-likelihood of the bivariate t copula with correlation rho and nu degrees of freedom."""
    x = stats.t.ppf(U, nu)
    q = (x[:, 0] ** 2 - 2 * rho * x[:, 0] * x[:, 1] + x[:, 1] ** 2) / (1 - rho ** 2)
    from scipy.special import gammaln
    lj = (gammaln((nu + 2) / 2) - gammaln(nu / 2) - np.log(nu * np.pi) - 0.5 * np.log(1 - rho ** 2)
          - (nu + 2) / 2 * np.log1p(q / nu))
    return float(np.sum(lj - stats.t.logpdf(x[:, 0], nu) - stats.t.logpdf(x[:, 1], nu)))


def gauss_copula_loglik(U, rho):
    x = stats.norm.ppf(U)
    q = (rho ** 2 * (x[:, 0] ** 2 + x[:, 1] ** 2) - 2 * rho * x[:, 0] * x[:, 1]) / (1 - rho ** 2)
    return float(np.sum(-0.5 * np.log(1 - rho ** 2) - 0.5 * q))


def t_tail_dependence(rho, nu):
    """lambda = 2 t_{nu+1}(-sqrt((nu + 1)(1 - rho)/(1 + rho))); the Gaussian copula has lambda = 0."""
    return float(2 * stats.t.cdf(-np.sqrt((nu + 1) * (1 - rho) / (1 + rho)), nu + 1))


def copula_fit(R):
    """rho from Kendall's tau (rho = sin(pi tau / 2)); nu of the t copula by maximum likelihood over a grid."""
    U = pseudo_obs(R)
    tau = float(stats.kendalltau(R.iloc[:, 0], R.iloc[:, 1])[0])
    rho = float(np.sin(np.pi * tau / 2))
    grid = np.arange(2.0, 40.01, 0.25)
    ll = np.array([t_copula_loglik(U, rho, v) for v in grid])
    nu = float(grid[np.argmax(ll)])
    return {'tau': tau, 'rho': rho, 'nu': nu, 'll_t': float(ll.max()), 'll_g': gauss_copula_loglik(U, rho),
            'lambda_t': t_tail_dependence(rho, nu), 'pearson': float(R.corr().iloc[0, 1]),
            'spearman': float(stats.spearmanr(R.iloc[:, 0], R.iloc[:, 1])[0])}


def empirical_tail_dep(R, qs=(0.05, 0.01)):
    """P(U1 <= q | U2 <= q): the share of the worst q of days of asset 2 that are also among the worst q of asset 1;
    under independence it equals q."""
    U = pseudo_obs(R)
    return {f'{q:g}': float(np.mean((U[:, 0] <= q) & (U[:, 1] <= q)) / q) for q in qs}


def simulate_copula(rho, nu=None, n=N_SIM, seed=SEED):
    """Uniform pairs from the Gaussian copula (nu None) or the t copula with correlation rho."""
    rng = np.random.default_rng(seed)
    Z = rng.multivariate_normal([0, 0], [[1, rho], [rho, 1]], n)
    if nu is None:
        return stats.norm.cdf(Z)
    w = np.sqrt(nu / rng.chisquare(nu, n))
    return stats.t.cdf(Z * w[:, None], nu)


def portfolio_analysis(R=None, w=WEIGHTS):
    """Portfolio VaR 1% and ES 2.5%: variance-covariance (Normal), historical simulation, and copula simulation with
    empirical margins (Gaussian and t copula); tail dependence."""
    R = portfolio_data() if R is None else R
    w = np.asarray(w)
    mu, S = R.mean().values, R.cov().values
    sd = np.sqrt(np.diag(S))
    rp = R.values @ w
    mp, sp = float(w @ mu), float(np.sqrt(w @ S @ w))
    cf = copula_fit(R)
    out = {'n': int(len(R)), 'first': R.index[0].date().isoformat(), 'mu': mu.tolist(), 'sd': sd.tolist(),
           'corr': float(R.corr().iloc[0, 1]), 'mp': mp, 'sp': sp,
           'var_n': normal_var(mp, sp), 'es_n': normal_es(mp, sp),
           'var_n_single': [normal_var(mu[i], sd[i]) for i in range(2)],
           'var_hs': hs_var(rp), 'es_hs': hs_es(rp), 'var_hs_single': [hs_var(R.values[:, i]) for i in range(2)],
           'copula': cf, 'tail_emp': empirical_tail_dep(R)}
    out['var_n_sum'] = float(w @ np.array(out['var_n_single']))
    out['var_hs_sum'] = float(w @ np.array(out['var_hs_single']))
    out['contrib'] = (w * (S @ w) / sp).tolist()               # Euler contributions to sigma_p
    for lab, nu in [('gauss', None), ('t', cf['nu'])]:
        U = simulate_copula(cf['rho'], nu)
        X = np.column_stack([np.quantile(R.values[:, i], U[:, i]) for i in range(2)])
        xp = X @ w
        q1 = [np.quantile(R.values[:, i], 0.01) for i in range(2)]
        out[f'var_{lab}'] = hs_var(xp)
        out[f'es_{lab}'] = hs_es(xp)
        out[f'joint1_{lab}'] = float(np.mean((X[:, 0] <= q1[0]) & (X[:, 1] <= q1[1])))
    q1 = [np.quantile(R.values[:, i], 0.01) for i in range(2)]
    out['joint1_emp'] = float(np.mean((R.values[:, 0] <= q1[0]) & (R.values[:, 1] <= q1[1])))
    out['joint1_ind'] = 0.0001
    for e, (a, b) in EPISODES.items():
        out[f'corr_{e}'] = float(R.loc[a:b].corr().iloc[0, 1])
    out['corr_calm'] = float(R.loc['2017-01-01':'2017-12-31'].corr().iloc[0, 1])
    return out


def fig_rolling_corr(R=None, save=True):
    """250-day rolling correlation of BET and S&P 500 daily returns (common days)."""
    R = portfolio_data() if R is None else R
    c = R.iloc[:, 0].rolling(250).corr(R.iloc[:, 1]).dropna()
    fig, ax = plt.subplots(figsize=(11, 3.6))
    for e, (a, b) in EPISODES.items():
        ax.axvspan(pd.Timestamp(a), pd.Timestamp(b), color=st.Amber, alpha=0.18, label='_')
    ax.plot(c.index, c, color=st.IDAred, lw=1.4, label='250-day correlation, BET and S&P 500')
    ax.axhline(R.corr().iloc[0, 1], color=st.MainBlue, ls='--', lw=1.2, label=f'whole sample: {R.corr().iloc[0, 1]:.2f}')
    ax.set_ylabel('correlation')
    st.legend_outside_bottom(ax, ncol=2, y=-0.15)
    st.check_no_grey(fig)
    if save:
        st.save_fig('sfm_ch10_rolling_corr')
    return {'max': float(c.max()), 'date_max': c.idxmax().date().isoformat(), 'min': float(c.min()),
            'last': float(c.iloc[-1])}


def fig_copula(R=None, fit=None, save=True):
    """Lower-left corner [0, 0.25]^2 of the pseudo-observations of BET and S&P 500 against samples of the same size from
    the fitted Gaussian and t copulas; the square marks the days on which both are among their worst 5%."""
    R = portfolio_data() if R is None else R
    cf = copula_fit(R) if fit is None else fit
    U = pseudo_obs(R)
    n = len(U)
    G = simulate_copula(cf['rho'], None, n, SEED + 1)
    Tt = simulate_copula(cf['rho'], cf['nu'], n, SEED + 2)
    fig, axes = plt.subplots(1, 3, figsize=(11.5, 4.0), sharey=True)
    q = 0.05
    for ax, (lab, X, c) in zip(axes, [('data (pseudo-observations)', U, st.MainBlue),
                                       (f'Gaussian copula, rho = {cf["rho"]:.2f}', G, st.Forest),
                                       (f't copula, rho = {cf["rho"]:.2f}, nu = {cf["nu"]:.1f}', Tt, st.IDAred)]):
        ax.scatter(X[:, 1], X[:, 0], s=7, color=c, alpha=0.6, label=lab)
        k = int(np.sum((X[:, 0] <= q) & (X[:, 1] <= q)))
        ax.plot([0, q, q], [q, q, 0], color=st.Orange, lw=2.0)
        ax.set_title(f'{k} pairs below 0.05 on both scales', fontsize=11)
        ax.set_xlabel('S&P 500 (uniform scale)')
        ax.set_xlim(0, 0.25)
        ax.set_ylim(0, 0.25)
    axes[0].set_ylabel('BET (uniform scale)')
    leg = st.fig_legend_bottom(fig, ncol=3, y=0.0)
    for h in leg.legend_handles:
        h.set_sizes([30])
        h.set_alpha(1)
    fig.tight_layout(rect=(0, 0.08, 1, 1))
    st.check_no_grey(fig)
    if save:
        st.save_fig('sfm_ch10_copula')
    return {'corner_data': int(np.sum((U[:, 0] <= q) & (U[:, 1] <= q))),
            'corner_gauss': int(np.sum((G[:, 0] <= q) & (G[:, 1] <= q))),
            'corner_t': int(np.sum((Tt[:, 0] <= q) & (Tt[:, 1] <= q))), 'n': int(n), 'expected_ind': float(n * q * q)}

## Run

In [ ]:
R = portfolio_data()
P = portfolio_analysis(R)
print({k: v for k, v in P.items() if k != 'copula'})
print(P['copula'])
print(fig_rolling_corr(R))
print(fig_copula(R, P['copula']))

![sfm_ch10_rolling_corr](./sfm_ch10_rolling_corr.png)

Output: `sfm_ch10_rolling_corr.pdf`

![sfm_ch10_copula](./sfm_ch10_copula.png)

Output: `sfm_ch10_copula.pdf`